# Spike 04 — PageIndex Reasoning Graph

**Goal:** Feed the OCR'd Markdown into PageIndex and build a reasoning graph (tree structure) over the document.

**Time box:** 1-2 hours

**Question to answer:** Does PageIndex successfully index the bilingual OCR output and return a usable reasoning structure?

**Done when:** Documents are indexed in PageIndex and we can query them without errors.

---

### What is Vector-Less Reasoning RAG?

Traditional RAG:
```
Document → chunk into pieces → embed as vectors → store in vector DB
Query    → embed query       → find nearest chunks → send to LLM
```
Problem: chunking breaks tables and cross-references. Arabic embeddings are weaker than English ones.

PageIndex approach:
```
Document → build reasoning graph (understands structure)
Query    → reasoning traversal over graph → find relevant pages
```
This is why it suits our bilingual, table-heavy urban reports.

### API Key Setup
1. Go to [pageindex.ai](https://pageindex.ai)
2. Click "Try Now" or "Book a Demo"
3. Mention you're in a hackathon — many services give free credits for this
4. Add to `.env`: `PAGEINDEX_API_KEY=your_key_here`

---
### ⚠️ Important
PageIndex's API details aren't fully public. This notebook uses the most likely API shape  
based on their documentation. **Adjust endpoint URLs and payload keys** based on what  
their team sends you after signup.

## Step 1 — Imports & Setup

In [ ]:
import requests
import json
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(dotenv_path="../.env")

PAGEINDEX_API_KEY = os.getenv("PAGEINDEX_API_KEY")
if not PAGEINDEX_API_KEY:
    raise ValueError("PAGEINDEX_API_KEY not found in .env — get it from pageindex.ai")

# Adjust this base URL to what PageIndex provides
BASE_URL = "https://api.pageindex.ai/v1"

HEADERS = {
    "Authorization": f"Bearer {PAGEINDEX_API_KEY}",
    "Content-Type": "application/json"
}

print("✅ PageIndex client configured")
print(f"   Base URL: {BASE_URL}")

## Step 2 — Verify Connection

In [ ]:
def check_connection() -> bool:
    """Ping PageIndex API to confirm key and connection are working."""
    try:
        r = requests.get(f"{BASE_URL}/health", headers=HEADERS, timeout=10)
        print(f"Status code : {r.status_code}")
        print(f"Response    : {r.text[:300]}")
        return r.status_code == 200
    except Exception as e:
        print(f"Connection error: {e}")
        print("Check the BASE_URL — adjust it based on PageIndex's actual docs")
        return False

check_connection()

## Step 3 — Load OCR Output

Read in the Markdown files produced by Spike 02 and 03.

In [ ]:
AR_DIR = Path("../ocr_output/arabic")
EN_DIR = Path("../ocr_output/english")

ar_files = sorted(AR_DIR.glob("*.md"))
en_files = sorted(EN_DIR.glob("*.md"))

print(f"Arabic pages  : {len(ar_files)}")
print(f"English pages : {len(en_files)}")

if not ar_files and not en_files:
    print("\n❌ No OCR output found")
    print("   Run Spike 02 for Arabic and Spike 03 for English first")
else:
    # Preview one file
    sample = (ar_files or en_files)[0]
    content = sample.read_text(encoding="utf-8")
    print(f"\nSample ({sample.name}): {len(content)} chars")
    print(content[:300])

## Step 4 — Upload a Single Document First (Test)

As always: test on ONE document before uploading everything.

In [ ]:
def upload_document(doc_id: str, content: str, language: str, metadata: dict = None) -> dict:
    """
    Upload a document to PageIndex.
    Adjust payload keys based on PageIndex's actual API schema.
    """
    payload = {
        "id"      : doc_id,
        "content" : content,
        "language": language,
        "metadata": metadata or {},
        "options" : {
            "build_reasoning_graph": True,   # the key PageIndex feature
            "index_tables"         : True,   # make sure tables are indexed
        }
    }

    r = requests.post(
        f"{BASE_URL}/documents",
        headers=HEADERS,
        json=payload,
        timeout=60
    )

    if r.status_code not in (200, 201):
        print(f"❌ Upload failed: {r.status_code} — {r.text[:500]}")
        return {}

    return r.json()


# Test with the first Arabic page
if ar_files:
    test_file    = ar_files[0]
    test_content = test_file.read_text(encoding="utf-8")
    test_id      = f"tranquil_ar_{test_file.stem}"

    print(f"Uploading test document: {test_id}")
    result = upload_document(
        doc_id   = test_id,
        content  = test_content,
        language = "ar",
        metadata = {
            "source": "Madinah Tranquil City 2024",
            "page"  : test_file.stem,
            "lang"  : "Arabic"
        }
    )
    print(f"\nResponse from PageIndex:")
    print(json.dumps(result, indent=2, ensure_ascii=False))

## Step 5 — Upload All Documents

In [ ]:
import time

uploaded = []
failed   = []

all_docs = [
    # (file, language, report_name)
    *[(f, "ar", "Madinah Tranquil City 2024 — Arabic")  for f in ar_files],
    *[(f, "en", "Madinah Tranquil City 2024 — English") for f in en_files],
]

for i, (file_path, lang, report_name) in enumerate(all_docs):
    doc_id  = f"tranquil_{lang}_{file_path.stem}"
    content = file_path.read_text(encoding="utf-8")

    print(f"[{i+1}/{len(all_docs)}] Uploading {doc_id}...", end=" ")

    result = upload_document(
        doc_id   = doc_id,
        content  = content,
        language = lang,
        metadata = {
            "source"  : report_name,
            "page"    : file_path.stem,
            "language": lang
        }
    )

    if result:
        print("✅")
        uploaded.append(doc_id)
    else:
        print("❌")
        failed.append(doc_id)

    time.sleep(0.5)  # be polite to the API

print(f"\n✅ Uploaded: {len(uploaded)}")
print(f"❌ Failed  : {len(failed)}")
if failed:
    print(f"   Failed IDs: {failed}")

## Step 6 — Inspect the Reasoning Graph

Ask PageIndex to return the graph/tree structure it built for a document.

In [ ]:
def get_reasoning_graph(doc_id: str) -> dict:
    """Retrieve the reasoning graph structure for an indexed document."""
    r = requests.get(
        f"{BASE_URL}/documents/{doc_id}/graph",
        headers=HEADERS,
        timeout=30
    )
    if r.status_code == 200:
        return r.json()
    else:
        print(f"❌ {r.status_code} — {r.text[:300]}")
        return {}


if uploaded:
    sample_id = uploaded[0]
    print(f"Fetching reasoning graph for: {sample_id}")
    graph = get_reasoning_graph(sample_id)

    print("\nReasoning Graph Structure:")
    print(json.dumps(graph, indent=2, ensure_ascii=False)[:2000])

## Step 7 — Spike Summary

In [ ]:
print("=" * 50)
print("SPIKE 04 — RESULTS")
print("=" * 50)
print(f"Total documents uploaded : {len(uploaded)}")
print(f"Arabic pages indexed     : {sum(1 for d in uploaded if '_ar_' in d)}")
print(f"English pages indexed    : {sum(1 for d in uploaded if '_en_' in d)}")
print(f"Failed uploads           : {len(failed)}")
print()

if len(uploaded) > 0 and len(failed) == 0:
    print("✅ SPIKE PASSED — Knowledge base ready for Spike 05 (retrieval)")
elif len(uploaded) > 0:
    print("⚠️  SPIKE PARTIAL — Some uploads failed, check error messages above")
else:
    print("❌ SPIKE FAILED — No documents uploaded")
    print("   Most likely cause: wrong BASE_URL or API schema")
    print("   Fix: Contact PageIndex support and get their exact API docs")